In [30]:
!pip install xgboost

  Using cached xgboost-3.1.2-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.6/72.0 MB 21.6 MB/s eta 0:00:04
   -- ------------------------------------- 5.2/72.0 MB 16.0 MB/s eta 0:00:05
   ---- ----------------------------------- 8.4/72.0 MB 16.3 MB/s eta 0:00:04
   ------ --------------------------------- 12.6/72.0 MB 16.4 MB/s eta 0:00:04
   ------------- -------------------------- 23.9/72.0 MB 24.4 MB/s eta 0:00:02
   ------------------ --------------------- 33.0/72.0 MB 27.6 MB/s eta 0:00:02
   ------------------------ --------------- 43.8/72.0 MB 31.3 MB/s eta 0:00:01
   ------------------------------ --------- 54.5/72.0 MB 34.1 MB/s eta 0:00:01
   --------------------------------- ------ 59.5/72.0 MB 33.3 MB/s eta 0:00:01
   ------------------------------------ --- 65.5/72.0 MB 32.1 MB/s eta 0:00:01
   ---------------------------------------- 72.0/72.0 MB 32.3 MB/s  0:00:

In [31]:
from functools import reduce

# --- PySpark ML และ Spark Libraries ---
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.evaluation import RegressionEvaluator
# 💡 แก้ไข Typo: SparkXGBRegresso -> SparkXGBRegressor
from xgboost.spark import SparkXGBRegressor 

# --- PySpark Types ---
from pyspark.sql.types import (
    ArrayType, StringType, StructType, StructField, 
    IntegerType, FloatType
)

# --- PySpark Functions (รวมฟังก์ชันทั้งหมด) ---
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, sum, udf, 
    to_timestamp, split, datediff, substring, length,
    current_timestamp, when, try_to_timestamp, to_date, 
    encode, decode, count, avg, min, max, lit, round
)

# --- External Libraries ---
from pythainlp import word_tokenize
from pythainlp.corpus import thai_stopwords

In [32]:
spark_url = 'local'
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder \
    .appName("XGBoostCondoPriceModel") \
    .config("spark.jars.packages", "ml.dmlc:xgboost4j-spark_2.12:1.7.0") \
    .getOrCreate()

In [33]:
sc = spark.sparkContext

# Path เดียวกันกับที่คุณใช้ในการบันทึก
OUTPUT_PATH_SUBSET = "file:///E:/2_1/ds/project/severity_output_jatujak.csv"

df_processed = spark.read.csv(
    OUTPUT_PATH_SUBSET, 
    header=True,
    encoding="UTF-8",
    # 💡 เพิ่ม inferSchema=True
    inferSchema=True 
)



In [34]:
df_processed = df_processed.drop("address", "lat","lon","comment_clean","timestamp_dt","last_activity_dt","year_reported","year_last_activity")
df_processed.printSchema() # ตรวจสอบอีกครั้ง

root
 |-- ticket_id: string (nullable = true)
 |-- district: string (nullable = true)
 |-- DaysActive_Pending: integer (nullable = true)
 |-- type_ถนน: integer (nullable = true)
 |-- type_ทางเท้า: integer (nullable = true)
 |-- type_ความปลอดภัย: integer (nullable = true)
 |-- type_แสงสว่าง: integer (nullable = true)
 |-- type_ความสะอาด: integer (nullable = true)
 |-- type_กีดขวาง: integer (nullable = true)
 |-- type_ท่อระบายน้ำ: integer (nullable = true)
 |-- type_น้ำท่วม: integer (nullable = true)
 |-- type_ต้นไม้: integer (nullable = true)
 |-- type_PM25: integer (nullable = true)
 |-- type_จราจร: integer (nullable = true)
 |-- type_สะพาน: integer (nullable = true)
 |-- predicted_severity: double (nullable = true)



In [38]:

CONDO_FILE_PATH = "ddproperty_processed.csv" 
# โปรดเปลี่ยนเป็น path จริงหากคุณไม่ได้อัปโหลดไฟล์ผ่านเครื่องมือ

df_condo_raw = spark.read.csv(
    CONDO_FILE_PATH,
    header=True,
    encoding="UTF-8",
    inferSchema=True  # ให้ Spark ลองเดาประเภทข้อมูล
)


df_condo_raw.printSchema()

root
 |-- url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- price_per_sqm: double (nullable = true)
 |-- usable_area: double (nullable = true)
 |-- bedroom: double (nullable = true)
 |-- restroom: double (nullable = true)
 |-- coords: string (nullable = true)
 |-- full_address: string (nullable = true)
 |-- sub_district: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



In [35]:
from pyspark.sql.functions import col, count, avg, sum

# 1. จัดกลุ่มด้วยเขต และคำนวณตัวชี้วัดปัญหา
df_district_metrics = df_processed.groupBy("district").agg(
    # A. Total Problem Count (ปริมาณปัญหาทั้งหมด)
    count(col("ticket_id")).alias("Total_Problem_Count"),
    
    # B. Average Severity (ความรุนแรงเฉลี่ย: ระยะเวลารอแก้ไข)
    avg(col("DaysActive_Pending")).alias("Avg_Pending_Days"),
    
    # C. Weighted Problem Type Index (ดัชนีปัญหาเฉพาะทาง - ให้ความปลอดภัยสำคัญสุด)
    # *หมายเหตุ: type_... ต้องถูก cast เป็น Int ก่อนแล้ว
    (
        (sum(col("type_ความปลอดภัย")) * 3) + 
        (sum(col("type_ทางเท้า")) * 2) +           
        (sum(col("type_น้ำท่วม")) * 3) +
        (sum(col("type_แสงสว่าง")) * 1)  +
        (sum(col("type_กีดขวาง")) * 2) +  
        (sum(col("type_ท่อระบายน้ำ")) * 2) +          
        (sum(col("type_ถนน")) * 2) +
        (sum(col("type_ต้นไม้")) * 1)+
        (sum(col("type_PM25")) * 2) +          
        (sum(col("type_จราจร")) * 2) +
        (sum(col("type_สะพาน")) * 1)      
    ).alias("Weighted_Problem_Index")
)

print("--- ตัวชี้วัดปัญหาต่อเขต (ก่อน Normalization) ---")
df_district_metrics.orderBy(col("Avg_Pending_Days").desc()).show(5, truncate=False)

--- ตัวชี้วัดปัญหาต่อเขต (ก่อน Normalization) ---
+--------+-------------------+-----------------+----------------------+
|district|Total_Problem_Count|Avg_Pending_Days |Weighted_Problem_Index|
+--------+-------------------+-----------------+----------------------+
|จตุจักร |5646               |714.8000354233085|16185                 |
+--------+-------------------+-----------------+----------------------+



In [36]:
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.sql.functions import lit, round

# 1. รวมคอลัมน์ปัญหาทั้งหมดเข้าด้วยกัน
feature_cols = ["Total_Problem_Count", "Avg_Pending_Days", "Weighted_Problem_Index"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_vector"
)
df_vector = assembler.transform(df_district_metrics)

# 2. ทำ Min-Max Scaling (ปรับค่าให้อยู่ในช่วง 0 ถึง 1)
scaler = MinMaxScaler(
    inputCol="features_vector", 
    outputCol="normalized_features"
)
scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

# 3. ดึงค่า Normalized กลับมาเป็นคอลัมน์ (ใช้ค่าแรกใน SparseVector)
# เนื่องจากเราไม่สามารถดึงค่าจาก Vector ใน PySpark ได้ง่าย ๆ เราจะใช้เทคนิคการ Scale ซ้ำ
# หรือในทางปฏิบัติที่ง่ายกว่าคือการคำนวณสูตร Min-Max ด้วยมือ (สำหรับตอนนี้)

In [37]:
from pyspark.sql.functions import (
    col, trim, regexp_replace, encode, decode, 
    count, avg, sum, min, max, lit, round, when
)
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.evaluation import RegressionEvaluator
from sparkdl.udf.keras_image_eval import SparkXGBoostRegressor # ต้องติดตั้ง spark-xgboost


# C. คำนวณตัวชี้วัดปัญหาต่อเขต (Metrics Aggregation)
df_district_metrics = df_ml_ready_filtered.groupBy("district").agg(
    count(col("ticket_id")).alias("Total_Problem_Count"),
    avg(col("predicted_severity")).alias("Avg_Severity_Score"),
    (
        (sum(col("type_ความปลอดภัย")) * 3) + 
        (sum(col("type_ทางเท้า")) * 2) +           
        (sum(col("type_น้ำท่วม")) * 3) +
        (sum(col("type_แสงสว่าง")) * 1)  +
        (sum(col("type_กีดขวาง")) * 2) +  
        (sum(col("type_ท่อระบายน้ำ")) * 2) +          
        (sum(col("type_ถนน")) * 2) +
        (sum(col("type_ต้นไม้")) * 1)+
        (sum(col("type_PM25")) * 2) +          
        (sum(col("type_จราจร")) * 2) +
        (sum(col("type_สะพาน")) * 1)       
    ).alias("Weighted_Problem_Index")
)

# D. Normalization และ Scoring
min_max_values = df_district_metrics.select(
    min(col("Total_Problem_Count")).alias("min_count"), max(col("Total_Problem_Count")).alias("max_count"),
    min(col("Avg_Severity_Score")).alias("min_severity"), max(col("Avg_Severity_Score")).alias("max_severity"),
    min(col("Weighted_Problem_Index")).alias("min_weight"), max(col("Weighted_Problem_Index")).alias("max_weight")
).collect()[0]

min_count, max_count = min_max_values["min_count"], min_max_values["max_count"]
min_severity, max_severity = min_max_values["min_severity"], min_max_values["max_severity"]
min_weight, max_weight = min_max_values["min_weight"], min_max_values["max_weight"]

df_scored = df_district_metrics.withColumn("Norm_Count", (col("Total_Problem_Count") - lit(min_count)) / lit(max_count - min_count)) \
                               .withColumn("Norm_Severity", (col("Avg_Severity_Score") - lit(min_severity)) / lit(max_severity - min_severity)) \
                               .withColumn("Norm_Weight", (col("Weighted_Problem_Index") - lit(min_weight)) / lit(max_weight - min_weight))

df_final_index = df_scored.withColumn(
    "Unlivability_Index",
    round((col("Norm_Count") * 0.3) + (col("Norm_Severity") * 0.4) + (col("Norm_Weight") * 0.3), 4)
)

# E. ผลลัพธ์: Livability Score เต็ม 10
df_final_score_new = df_final_index.withColumn(
    "Livability_Score_10",
    round((lit(1.0) - col("Unlivability_Index")) * 10, 2)
)

print(f"✅ จำนวนเขตที่ใช้คำนวณ Livability Score: {df_final_score_new.count()} เขต")
df_final_score_new.select("district", "Livability_Score_10").orderBy(col("Livability_Score_10").desc()).show(5, truncate=False)


ModuleNotFoundError: No module named 'sparkdl'

In [ ]:
print("--- 3. รวม Livability Score เข้ากับข้อมูลคอนโด ---")

df_livability_features = df_final_score_new.select(
    col("district"), 
    col("Livability_Score_10")
)

# ทำ Left Join เพื่อเชื่อม Livability Score เข้ากับคอนโด
df_condo_with_livability = df_condo_clean_final.join(
    df_livability_features, 
    on="district", 
    how="left"
)




{"ts": "2025-12-06 18:55:50.761", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012", "context": {"file": "line 22 in cell [14]", "line": "", "fragment": "__truediv__", "errorClass": "DIVIDE_BY_ZERO"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o816.showString.\n: org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012\n== DataFrame ==\n\"__truediv__\" was called from\nline 22 in cell [14]\n\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:203)\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__truediv__" was called from
line 22 in cell [14]


In [ ]:

# 1. แบ่งข้อมูล
train_data, test_data = df_final_ml.randomSplit([0.8, 0.2], seed=42)

# 2. กำหนดโมเดล XGBoost Regressor
xgb = SparkXGBoostRegressor(
    featuresCol="features", 
    labelCol="label",
    num_workers=2, 
    max_depth=6, 
    n_estimators=100,
    learning_rate=0.1
)

# 3. เทรนโมเดล
xgb_model = xgb.fit(train_data)

# 4. ประเมินผล
predictions = xgb_model.transform(test_data)
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("\n--- ผลการประเมินโมเดล XGBoost ---")
print(f"Root Mean Squared Error (RMSE): {rmse:,.2f}")
print(f"R-squared (R²): {r2:.4f}")
print("-------------------------------------------------------")